# 01 — Exploratory Data Analysis

This notebook connects to the raw Divvy trip parquet file via DuckDB and performs
an initial schema inspection. It examines null counts, data types, and categorical
distributions to build a working understanding of the dataset before any feature
engineering is applied.

## Assumptions & Dependencies

- The raw parquet file (`divvy.parquet`) must exist at the path provided when prompted.
- No prior notebook needs to have been run — this is the first stage in the pipeline.
- Required packages: `duckdb`, `pandas`, `matplotlib` (see `requirements.txt`).

## Setup — load data path and connect DuckDB

We prompt for the file path at runtime so the notebook works on any machine
without hardcoded paths. The `connect_duckdb` helper registers the parquet file
as a DuckDB view named `trips`.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils import get_data_path, connect_duckdb

file_path = get_data_path()
con = connect_duckdb(file_path)

## Schema Inspection

Inspect column names, data types, and the total row count to confirm the parquet
file loaded correctly and matches expectations.

In [ ]:
# Schema and row count
print(con.execute("DESCRIBE trips").df())
print("\nRow count:", con.execute("SELECT COUNT(*) FROM trips").fetchone()[0])

## Null Counts

Count nulls per column to identify which fields will need imputation or special
handling in the feature engineering stage.

In [ ]:
# Null counts per column
null_query = """
SELECT
    COUNT(*) - COUNT(trip_id)          AS trip_id_nulls,
    COUNT(*) - COUNT(starttime)        AS starttime_nulls,
    COUNT(*) - COUNT(stoptime)         AS stoptime_nulls,
    COUNT(*) - COUNT(from_station_id)  AS from_station_id_nulls,
    COUNT(*) - COUNT(to_station_id)    AS to_station_id_nulls,
    COUNT(*) - COUNT(temperature)      AS temperature_nulls,
    COUNT(*) - COUNT(events)           AS events_nulls
FROM trips
"""
con.execute(null_query).df()

## Categorical Distributions

Examine the distribution of key categorical fields — station IDs, weather events,
and user types — to understand cardinality and class balance.

In [ ]:
# Station count and trip date range
print(con.execute("SELECT COUNT(DISTINCT from_station_id) AS station_count FROM trips").df())
print(con.execute("SELECT MIN(starttime)::DATE AS min_date, MAX(starttime)::DATE AS max_date FROM trips").df())

In [ ]:
# Weather events distribution
con.execute("SELECT events, COUNT(*) AS trip_count FROM trips GROUP BY events ORDER BY trip_count DESC").df()

## Summary

**Outputs produced by this notebook:** None — this is a read-only exploration.

**What the next notebook expects:** `02_feature_engineering.ipynb` assumes the
same parquet file is accessible at the path returned by `get_data_path()` and
that the `trips` view schema has been confirmed here.